# Walk through a v4 reconstruction
Run these cells with the project environment. The notebook imports the same network and preprocessing used by the local UI. The example uses the first fixed FFHQ test face. Run the FFHQ setup in the README first.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
import hashlib
import json
import numpy as np
from PIL import Image
import torch
from face_sr import runtime
from input_quality import InputDegradation

## 1. Load the released weights
A strict match checks every parameter name and shape. The SHA-256 digest identifies the validation-selected checkpoint.

In [ ]:
review = json.loads((root / 'work/recognition_v4/review.json').read_text())
path = root / review['checkpoint']
assert hashlib.sha256(path.read_bytes()).hexdigest() == review['checkpoint_sha256']
checkpoint = torch.load(path, map_location='cpu', weights_only=True)
model = runtime.SRNet(runtime.Config()).eval()
model.load_state_dict(checkpoint['model'], strict=True)
print(f'{runtime.n_params(model):,} learned parameters')

## 2. Prepare the observation
The UI uses this center crop and degradation seed for test face 01. Keep the clean reference for evaluation.

In [ ]:
faces = np.load(root / 'work/data/ffhq128_u8.npy', mmap_mode='r')
with np.load(root / 'work/full/split_ids.npz') as ids:
    original = faces[ids['test'][0], 8:120, 8:120].copy()
low = InputDegradation(runtime, 'standard28')(original, np.random.default_rng(2000))
def tensor(rgb):
    return torch.from_numpy(rgb.copy()).permute(2, 0, 1).float().unsqueeze(0) / 255
display(Image.fromarray(low))

## 3. Reconstruct the face
The head extracts 48 feature channels. Six residual blocks combine local samples and attention. Two PixelShuffle layers enlarge the feature map, and a bicubic skip carries input structure into the result.

In [ ]:
with torch.inference_mode():
    output = model(tensor(low)).clamp(0, 1)
rgb = output[0].permute(1, 2, 0).mul(255).round().byte().numpy()
display(Image.fromarray(rgb), Image.fromarray(original))

## 4. Measure agreement with the reference
The UI measures luminance PSNR and SSIM after removing a four-pixel border. The recognition evaluation uses labeled camera pairs separately.

In [ ]:
target = tensor(original)
print('PSNR-Y (dB)', runtime.Metrics.psnr_y(output, target, 4).item())
print('SSIM-Y', runtime.Metrics.ssim_y(output, target, 4).item())